In [ ]:
from datasets import load_dataset

In [ ]:
papers_data = load_dataset('csv', data_files='./data/redacoes.csv')

In [ ]:
papers_data

In [ ]:
papers_data['train'].features

In [ ]:
papers_data['train'].to_pandas()

In [ ]:
train_test = papers_data['train'].train_test_split(test_size=0.2, shuffle=False)
train_test

In [ ]:
from datasets import DatasetDict

In [ ]:
from transformers import AutoTokenizer
base_model = 'Geotrend/distilbert-base-pt-cased'
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [ ]:
def tokenizer_data(text_data):
    return tokenizer(text_data['essay'], truncation=True)

In [ ]:
papers_data

In [ ]:
tokenized_dataset = train_test.map(tokenizer_data, batched=True, remove_columns=['essay'])

In [ ]:
tokenized_dataset

In [ ]:
from datasets import ClassLabel

In [ ]:
scores = ClassLabel(names=[str(i) for i in range(11)])

In [ ]:
def map_labels(data):
    data['label'] = scores.str2int(str(data['score']))
    return data

In [ ]:
tokenized_dataset = tokenized_dataset.map(map_labels, remove_columns=['score'])

In [ ]:
tokenized_dataset['train'].features['label']

In [ ]:
tokenized_dataset = tokenized_dataset.cast_column('label', scores)

In [ ]:
tokenized_dataset['train'].features['label']

In [ ]:
# Importa as classes necessárias para carregar o tokenizador e o modelo de classificação
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# Define os mapeamentos entre índice numérico e rótulo textual (notas de 0 a 10)
id2label = {i:str(i) for i in range(11)}
label2id = {value: key for key, value in id2label.items()}

In [ ]:
# Carrega o tokenizador e o modelo pré-treinado mDeBERTa multilingual.
# O modelo original foi treinado para NLI com 3 classes; aqui substituímos
# a camada de classificação por uma nova com 11 saídas (notas 0-10).
# ignore_mismatched_sizes=True permite essa substituição sem erro.
model_name = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=tokenized_dataset['train'].features['label'].num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

In [ ]:
# Define os hiperparâmetros de treinamento e cria o otimizador e o scheduler.
# AdamW é o otimizador padrão para fine-tuning de transformers.
# O scheduler reduz a taxa de aprendizado linearmente até zero ao longo do treino.
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

batch_size = 16
epochs = 2
batches_per_epoch = len(tokenized_dataset['train']) // batch_size
total_training_steps = int(batches_per_epoch * epochs)
learning_rate = 2e-5

optimizer = AdamW(model.parameters(), lr=learning_rate)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_training_steps
)

In [ ]:
# Configura e executa o fine-tuning do modelo usando a API Trainer do HuggingFace.
# O Trainer gerencia o loop de treinamento, avaliação por época e os logs automaticamente.
# use_cpu=True é necessário pois a GPU disponível (920MX) não é suportada pelo PyTorch 2.x.
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

# DataCollatorWithPadding aplica padding dinâmico em cada batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_strategy="epoch",
    use_cpu=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator,
    optimizers=(optimizer, scheduler),
)

trainer.train()

In [ ]:
# Salva o modelo fine-tunado e o tokenizador em disco.
# Isso permite reutilizar o modelo em outros projetos sem retreinar.
model.save_pretrained("./essay-scorer")
tokenizer.save_pretrained("./essay-scorer")

In [ ]:
# Avalia o modelo nos dados de teste.
# trainer.predict() retorna os logits brutos; np.argmax seleciona a classe com maior pontuação.
# Exibe a acurácia geral e compara previsão vs. rótulo real nas primeiras 20 amostras.
import numpy as np

predictions = trainer.predict(tokenized_dataset['test'])
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

accuracy = (pred_labels == true_labels).mean()
print(f"Accuracy: {accuracy:.4f}")

print("\nPredicted vs True (first 20 samples):")
for pred, true in zip(pred_labels[:20], true_labels[:20]):
    match = "✓" if pred == true else "✗"
    print(f"  {match}  predicted={pred}  true={true}")